In [7]:
# polars parctice topics 
import polars as pl
data = {
    "Name":["alice","bob","Abhinav","purav","satyam"],
    "Age" :[23, 30, 25, 35, 28],
    "Price" :[100,200,150,300,250]
}
df = pl.DataFrame(data)
print(df)
# selecting columms 
print(df.select("Name"))
print(df.select(["Age"]))
print(df.select(["Price"]))

shape: (5, 3)
┌─────────┬─────┬───────┐
│ Name    ┆ Age ┆ Price │
│ ---     ┆ --- ┆ ---   │
│ str     ┆ i64 ┆ i64   │
╞═════════╪═════╪═══════╡
│ alice   ┆ 23  ┆ 100   │
│ bob     ┆ 30  ┆ 200   │
│ Abhinav ┆ 25  ┆ 150   │
│ purav   ┆ 35  ┆ 300   │
│ satyam  ┆ 28  ┆ 250   │
└─────────┴─────┴───────┘
shape: (5, 1)
┌─────────┐
│ Name    │
│ ---     │
│ str     │
╞═════════╡
│ alice   │
│ bob     │
│ Abhinav │
│ purav   │
│ satyam  │
└─────────┘
shape: (5, 1)
┌─────┐
│ Age │
│ --- │
│ i64 │
╞═════╡
│ 23  │
│ 30  │
│ 25  │
│ 35  │
│ 28  │
└─────┘
shape: (5, 1)
┌───────┐
│ Price │
│ ---   │
│ i64   │
╞═══════╡
│ 100   │
│ 200   │
│ 150   │
│ 300   │
│ 250   │
└───────┘


In [5]:
import polars as pl
import numpy as np

data = {
    "Name": ["Alice", "Bob", "Abhinav", "Purav", "Satyam"],
    "Age": [23, 30, 25, 35, None],
    "Price": [100, 200, 150, 300, 250],
    "Dept": ["IT", "HR", "IT", "Sales", "HR"]
}

df = pl.DataFrame(data)

# 1. Null Handling
df = df.with_columns(pl.col("Age").fill_null(df["Age"].mean()))

# 2. Filtering
df_filter = df.filter(pl.col("Age") > 25)

# 3. Sorting
df_sort = df.sort("Price", descending=True)

# 4. Feature Engineering
df = df.with_columns((pl.col("Price") * 0.9).alias("Discounted_Price"))

# 5. Groupby & Aggregation
summary = df.group_by("Dept").agg(
    pl.col("Price").mean().alias("Avg_Price"),
    pl.col("Name").count().alias("Count")
)

# 6. Basic EDA
print(df.describe())
print(df.null_count())

print("\nFiltered:\n", df_filter)
print("\nSummary:\n", summary)

shape: (9, 6)
┌────────────┬─────────┬──────────┬───────────┬───────┬──────────────────┐
│ statistic  ┆ Name    ┆ Age      ┆ Price     ┆ Dept  ┆ Discounted_Price │
│ ---        ┆ ---     ┆ ---      ┆ ---       ┆ ---   ┆ ---              │
│ str        ┆ str     ┆ f64      ┆ f64       ┆ str   ┆ f64              │
╞════════════╪═════════╪══════════╪═══════════╪═══════╪══════════════════╡
│ count      ┆ 5       ┆ 5.0      ┆ 5.0       ┆ 5     ┆ 5.0              │
│ null_count ┆ 0       ┆ 0.0      ┆ 0.0       ┆ 0     ┆ 0.0              │
│ mean       ┆ null    ┆ 28.25    ┆ 200.0     ┆ null  ┆ 180.0            │
│ std        ┆ null    ┆ 4.656984 ┆ 79.056942 ┆ null  ┆ 71.151247        │
│ min        ┆ Abhinav ┆ 23.0     ┆ 100.0     ┆ HR    ┆ 90.0             │
│ 25%        ┆ null    ┆ 25.0     ┆ 150.0     ┆ null  ┆ 135.0            │
│ 50%        ┆ null    ┆ 28.25    ┆ 200.0     ┆ null  ┆ 180.0            │
│ 75%        ┆ null    ┆ 30.0     ┆ 250.0     ┆ null  ┆ 225.0            │
│ max      

In [2]:
import polars as pl

df = pl.DataFrame(
    {
        "Name": ["Alice", "Bob", "Abhinav", "Purav", "Satyam", "Bob", "Alice"],
        "Age": [23.0, 30.0, 25.0, 35.0, None, 30.0, 23.0],
        "Price": [100.0, 200.0, 150.0, 300.0, 250.0, None, 120.0],
        "Dept": ["IT", "HR", "IT", "Sales", "HR", "HR", "IT"],
        "City": ["Delhi", "Mumbai", None, "Pune", "Delhi", "Mumbai", "Delhi"],
    }
)

print(df)
print(df.shape)
print(df.head(3))
print(df.tail(3))
print(df.columns)
print(df.dtypes)

print(df.describe())
print(df.null_count())

df = df.with_columns(
    pl.col("Age").fill_null(pl.col("Age").median()),
    pl.col("Price").fill_null(pl.col("Price").median()),
    pl.col("City").fill_null("Missing"),
)

print(df.select(["Name", "Age"]))
print(df.filter(pl.col("Age") > 25))
print(df.filter((pl.col("Dept") == "HR") & (pl.col("Price") > 150)))

print(df.sort("Price", descending=True))
print(df.sort(["Dept", "Age"]))

df = df.with_columns(
    (pl.col("Price") * 0.9).alias("DiscountedPrice"),
    (pl.col("Age") + pl.col("Price")).alias("AgePlusPrice"),
    pl.col("Name").str.to_uppercase().alias("NameUpper"),
)

print(df)

summary = df.group_by("Dept").agg(
    pl.len().alias("Count"),
    pl.col("Price").mean().alias("AvgPrice"),
    pl.col("Price").max().alias("MaxPrice"),
    pl.col("Age").median().alias("MedianAge"),
).sort("AvgPrice", descending=True)

print(summary)

df_city = pl.DataFrame(
    {"City": ["Delhi", "Mumbai", "Pune", "Missing"], "State": ["DL", "MH", "MH", "NA"]}
)

joined = df.join(df_city, on="City", how="left")
print(joined)

print(df.unique(subset=["Name", "Dept"]))
print(df.select(pl.col("Name").n_unique().alias("unique_names")))

print(df.select(pl.col("Age").quantile(0.25).alias("q1_age"), pl.col("Age").quantile(0.75).alias("q3_age")))

df = df.with_columns(
    pl.when(pl.col("Price") >= 200).then("High").otherwise("Low").alias("PriceLevel")
)

print(df.select(["Name", "Price", "PriceLevel"]))

df = df.to_dummies(columns=["Dept", "PriceLevel"], separator="_")
print(df)

df.write_csv("polars_practice_output.csv")
df2 = pl.read_csv("polars_practice_output.csv")
print(df2.head(2))

lazy_result = (
    df2.lazy()
    .filter(pl.col("Price") > 100)
    .group_by("City")
    .agg(pl.col("Price").mean().alias("MeanPrice"))
    .sort("MeanPrice", descending=True)
    .collect(streaming=True)
)

print(lazy_result)

shape: (7, 5)
┌─────────┬──────┬───────┬───────┬────────┐
│ Name    ┆ Age  ┆ Price ┆ Dept  ┆ City   │
│ ---     ┆ ---  ┆ ---   ┆ ---   ┆ ---    │
│ str     ┆ f64  ┆ f64   ┆ str   ┆ str    │
╞═════════╪══════╪═══════╪═══════╪════════╡
│ Alice   ┆ 23.0 ┆ 100.0 ┆ IT    ┆ Delhi  │
│ Bob     ┆ 30.0 ┆ 200.0 ┆ HR    ┆ Mumbai │
│ Abhinav ┆ 25.0 ┆ 150.0 ┆ IT    ┆ null   │
│ Purav   ┆ 35.0 ┆ 300.0 ┆ Sales ┆ Pune   │
│ Satyam  ┆ null ┆ 250.0 ┆ HR    ┆ Delhi  │
│ Bob     ┆ 30.0 ┆ null  ┆ HR    ┆ Mumbai │
│ Alice   ┆ 23.0 ┆ 120.0 ┆ IT    ┆ Delhi  │
└─────────┴──────┴───────┴───────┴────────┘
(7, 5)
shape: (3, 5)
┌─────────┬──────┬───────┬──────┬────────┐
│ Name    ┆ Age  ┆ Price ┆ Dept ┆ City   │
│ ---     ┆ ---  ┆ ---   ┆ ---  ┆ ---    │
│ str     ┆ f64  ┆ f64   ┆ str  ┆ str    │
╞═════════╪══════╪═══════╪══════╪════════╡
│ Alice   ┆ 23.0 ┆ 100.0 ┆ IT   ┆ Delhi  │
│ Bob     ┆ 30.0 ┆ 200.0 ┆ HR   ┆ Mumbai │
│ Abhinav ┆ 25.0 ┆ 150.0 ┆ IT   ┆ null   │
└─────────┴──────┴───────┴──────┴────────┘
shape:

ColumnNotFoundError: unable to find column "High"; valid columns: ["Name", "Age", "Price", "Dept", "City", "DiscountedPrice", "AgePlusPrice", "NameUpper"]

In [ ]:
import polars as pl

data = {
    "Name": ["Alice", "Bob", "Abhinav", "Purav", "Satyam"],
    "Age": [23.0, 30.0, 25.0, 35.0, None],
    "Price": [100.0, 200.0, 150.0, 300.0, 250.0],
    "Dept": ["IT", "HR", "IT", "Sales", "HR"]
}
df = pl.DataFrame(data)

# 1. Basic EDA
print("--- EDA ---")
print(df.describe())
print("Null counts:", df.null_count())

# 2. Null Handling
df = df.with_columns(pl.col("Age").fill_null(df["Age"].median()))

# 3. Selection & Filtering
df_subset = df.select(["Name", "Age"]).filter(pl.col("Age") > 25)

# 4. Sorting
df = df.sort("Price", descending=True)

# 5. Feature Engineering
df = df.with_columns((pl.col("Price") * 0.9).alias("Discounted_Price"))

# 6. Encoding
df = df.to_dummies(columns=["Dept"])

# 7. Groupby & Aggregation
summary = df.group_by("Dept_IT").agg([
    pl.col("Price").mean().alias("Mean_Price"),
    pl.col("Age").max().alias("Max_Age")
])

# 8. CSV Save/Read
df.write_csv("output.csv")
new_df = pl.read_csv("output.csv")

print("\n--- Final Data ---")
print(df)
print("\n--- Summary ---")
print(summary)

--- EDA ---
shape: (9, 5)
┌────────────┬─────────┬──────────┬───────────┬───────┐
│ statistic  ┆ Name    ┆ Age      ┆ Price     ┆ Dept  │
│ ---        ┆ ---     ┆ ---      ┆ ---       ┆ ---   │
│ str        ┆ str     ┆ f64      ┆ f64       ┆ str   │
╞════════════╪═════════╪══════════╪═══════════╪═══════╡
│ count      ┆ 5       ┆ 4.0      ┆ 5.0       ┆ 5     │
│ null_count ┆ 0       ┆ 1.0      ┆ 0.0       ┆ 0     │
│ mean       ┆ null    ┆ 28.25    ┆ 200.0     ┆ null  │
│ std        ┆ null    ┆ 5.377422 ┆ 79.056942 ┆ null  │
│ min        ┆ Abhinav ┆ 23.0     ┆ 100.0     ┆ HR    │
│ 25%        ┆ null    ┆ 25.0     ┆ 150.0     ┆ null  │
│ 50%        ┆ null    ┆ 30.0     ┆ 200.0     ┆ null  │
│ 75%        ┆ null    ┆ 30.0     ┆ 250.0     ┆ null  │
│ max        ┆ Satyam  ┆ 35.0     ┆ 300.0     ┆ Sales │
└────────────┴─────────┴──────────┴───────────┴───────┘
Null counts: shape: (1, 4)
┌──────┬─────┬───────┬──────┐
│ Name ┆ Age ┆ Price ┆ Dept │
│ ---  ┆ --- ┆ ---   ┆ ---  │
│ u32  ┆ u32 ┆ u3

In [9]:
import polars as pl
data = {
    "Name":["alice","bob","Abhinav","purav","satyam"],
    "Age" :[23, 30, 25, 35, 28],
    "Price" :[100,200,150,300,250]
}
df = pl.DataFrame(data)
print(df)

shape: (5, 3)
┌─────────┬─────┬───────┐
│ Name    ┆ Age ┆ Price │
│ ---     ┆ --- ┆ ---   │
│ str     ┆ i64 ┆ i64   │
╞═════════╪═════╪═══════╡
│ alice   ┆ 23  ┆ 100   │
│ bob     ┆ 30  ┆ 200   │
│ Abhinav ┆ 25  ┆ 150   │
│ purav   ┆ 35  ┆ 300   │
│ satyam  ┆ 28  ┆ 250   │
└─────────┴─────┴───────┘
